# What has to be true before you're allowed to just add things up

**Paper:** *Projection Obstruction for Factorized States* (PO-000, v6.3)

`AuthorityEffect: None.` Nothing here authorises any decision.

---

## The question, with no mathematics

You have a system with several parts, and a way of scoring how much is wrong
in each part. You want one number for the whole system.

The obvious move is to add the parts together. Almost everyone does this
without thinking about it.

**When is that actually justified?** Not "when is it convenient" — when is it
*forced*, so that adding up is the only scoring rule consistent with your
assumptions?

That is the question this part of the paper answers. It lists five conditions
(called RE1–RE5) and shows that if you assume them, plain addition is what you
get. Not approximately. Exactly.

## Why this is more than pedantry

Scoring rules that *look* reasonable but are not additive are everywhere. The
most familiar is the ordinary distance formula: for legs of length 3 and 4, the
straight-line combination gives 5, not 7. That is the right answer for
geometry, and the wrong answer if you wanted a total.

So "combine the parts sensibly" is not enough to pin down addition. Something
stronger has to be assumed — and the paper's contribution is saying exactly
what, and then showing that dropping any of it really does break the result.

## The five conditions in plain words

| Name | Plain meaning |
| --- | --- |
| **RE1** | the whole-system score is a genuine gauge: zero only at zero, proportional under scaling, and subadditive |
| **RE2** | the score depends only on each declared *local magnitude* — not on direction or phase. Labels may still matter |
| **RE3** | scores on disjoint groups add |
| **RE4** | declared type-preserving exchanges do not change the score |
| **RE5** | a one-part score uses exactly the declared local gauge, fixing the units |

RE3 is the one doing the heavy lifting, and RE5 is what fixes the units. In
fact those two are enough on their own: **once RE3 and RE5 are assumed, RE1,
RE2 and RE4 are redundant** for the unweighted conclusion.

## What we will do

1. Check that ordinary addition satisfies all five.
2. Drop RE3 and watch additivity fail.
3. Drop RE5 and watch the units drift.

Steps 2 and 3 are the important ones. Showing a rule *works* is easy; showing
that an assumption is **load-bearing** — that removing it lets something false
through — is what makes the result content rather than decoration.

Only two of the five carry surviving one-drop countermodels: RE3 and RE5. That
is not a weakness of the result, it is the result: RE3 and RE5 alone force the
unweighted sum, and the other three are redundant given them. This notebook
demonstrates the two drops that bite, and does not claim the other three are
independently necessary.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str((Path.cwd() if (Path.cwd() / "code").exists() else Path.cwd().parent).resolve() / "code"))

from exactsym import Rational
from replica_hierarchy import (
    FiniteDiagnosticFamily,
    absolute_gauge,
    lp_counterexample_diagnostic,
    scaled_sum_counterexample,
    unweighted_l1_diagnostic,
)

AXIOMS = (
    ("re1_global_gauge", "RE1  same yardstick everywhere"),
    ("re2_local_magnitude_sufficiency", "RE2  only size matters"),
    ("re3_disjoint_replica_additivity", "RE3  separate problems add"),
    ("re4_edge_exchange_symmetry", "RE4  swapping parts changes nothing"),
    ("re5_singleton_calibration", "RE5  one unit scores one"),
)

GRID = {
    "e1": [(Rational(0),), (Rational(1),)],
    "e2": [(Rational(0),), (Rational(1),)],
}


def check(scoring_rule, label):
    """Test one scoring rule against all five conditions and print the result."""
    family = FiniteDiagnosticFamily(
        edges=["e1", "e2"],
        gauges={"e1": absolute_gauge, "e2": absolute_gauge},
        diagnostic=scoring_rule,
    )
    audit = family.audit(samples=GRID)
    print(label)
    print(f"  (checked {audit.checked_points} points exactly, no floating point)")
    for attribute, description in AXIOMS:
        holds = getattr(audit, attribute)
        print(f"    {'HOLDS ' if holds else 'FAILS '}  {description}")
    return audit


print("loaded replica_hierarchy from code/")

## Step 1 — ordinary addition

The honest scoring rule: add up the sizes. All five conditions should hold.

In [ ]:
honest = check(unweighted_l1_diagnostic, "Adding the parts up")

## Step 2 — drop RE3 and additivity really does fail

Now the straight-line distance rule from the introduction: for two separate
problems of size 1 each, it scores about 1.41 rather than 2.

This is a perfectly respectable way to combine numbers. It is simply **not**
additive — which is exactly what RE3 forbids.

In [ ]:
broken = check(lp_counterexample_diagnostic, "Straight-line (Euclidean) combination")

print()
print("RE3 held for plain addition :", honest.re3_disjoint_replica_additivity)
print("RE3 held for this rule      :", broken.re3_disjoint_replica_additivity)
print()
print("So RE3 is load-bearing: a sensible-looking rule violates it.")

## Step 3 — drop RE5 and the units drift

A subtler failure. This rule *is* additive — separate problems still add
correctly — but it multiplies everything by a constant. Every relative
comparison it makes is right; every absolute number it reports is wrong by a
fixed factor.

RE5 is what rules this out, by nailing down that one unit of problem scores
exactly one. It is the difference between a score you can compare and a score
you can quote.

In [ ]:
for factor in ("2", "3", "1/2"):
    audit = check(
        scaled_sum_counterexample(Rational(factor)),
        f"Addition, but everything multiplied by {factor}",
    )
    print(f"    -> still additive: {audit.re3_disjoint_replica_additivity}"
          f"  |  correctly calibrated: {audit.re5_singleton_calibration}")
    print()

## What this establishes

Adding up the parts is not the only conceivable scoring rule, and it is not
obviously the right one. It becomes the right one once RE3 and RE5 are assumed;
RE1, RE2 and RE4 are then redundant for the unweighted conclusion. Dropping
either RE3 or RE5 lets a concrete alternative through, which is what the two
steps above exhibit.

In the paper this is Theorem 4.2 and the countermodels of §4.5. The algebraic
core is machine-checked in Lean 4 (`po000_l005_weighted_hierarchy.lean`).

## What this does not say

- The five conditions are **declared**, not derived. The paper is explicit that
  they are structural observer declarations you choose to adopt, not facts
  about the world.
- An earlier version of the paper claimed all five were independent. That claim
  was **withdrawn**: only the RE3-drop and RE5-drop countermodels survive, and
  those two are exactly what this notebook demonstrates.
- Running this notebook checks finitely many points. The general theorem is
  proved in the paper, not here.

That withdrawal is recorded in the claim ledger rather than quietly dropped,
which is the discipline the whole deposit runs on.